In [17]:
import re
import numpy as np
import nltk
from collections import defaultdict, Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

# Download the NLTK resources required by this notebook.
# Existing resources are reused, so these calls are safe to run again.
for resource in [
    "punkt",
    "punkt_tab",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger_eng",
]:
    nltk.download(resource, quiet=True)


In [18]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def lemmatize_words(words):
    tagged = pos_tag(words)
    return [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in tagged]

In [ ]:
def preprocess_with_boundaries(text, use_lemmatization=True):

    # Remove only tags, keep the text inside them
    text = re.sub(r'<[^>]+>', ' ', text)

    text = text.lower()

    sentences = sent_tokenize(text)

    flattened_tokens = []

    for sentence in sentences:

        sentence = re.sub(r'[^a-z\s]', ' ', sentence)
        sentence = re.sub(r'\s+', ' ', sentence).strip()

        if not sentence:
            continue

        words = word_tokenize(sentence)

        if use_lemmatization:
            words = lemmatize_words(words)

        if words:
            flattened_tokens.extend(
                ['<s>'] + words + ['</s>']
            )

    return flattened_tokens



def load_corpus_tokens(use_lemmatization=True):
    filepath = "corpus.txt"
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    return preprocess_with_boundaries(text, use_lemmatization)



def build_ngram_model(tokens, n):
    counts = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i:i + n - 1])
        next_word = tokens[i + n - 1]               #for corpus <s> i am sam </s> <s> i am hungry </s> =>
        counts[history][next_word] += 1             #counts contains => history:('i','am') , counter(Counter({'sam' : 1 , 'hungry' : 1 })) 

    prob_model = defaultdict(dict)
    for history, counter in counts.items():         #returns (('i','am'), Counter({'sam': 1, 'hungry': 1}) .... history, counter
        total = sum(counter.values())               #counter.keys() => ['sam', 'hungry']     counter.values() => [1, 1]     counter.items() => [('sam', 1), ('hungry', 1)]
        for word, count in counter.items():
            prob_model[history][word] = count / total  

    return prob_model                               #returns probablity of all [history][next word]


def train_models():
    raw_tokens = load_corpus_tokens(use_lemmatization=False)
    lemma_tokens = load_corpus_tokens(use_lemmatization=True)

    raw_models = {
        4: build_ngram_model(raw_tokens, n=4),      #{4 : [ [[h][nw] , probablity] , [[h][nw] , probablity] .... ]}      4: { ('i','am','sam'): {'hungry': 0.6, '</s>': 0.4}, ... },
        3: build_ngram_model(raw_tokens, n=3),                                                                        #  3: { ('i','am'): {'sam': 0.7, 'hungry': 0.3}, ... },
        2: build_ngram_model(raw_tokens, n=2),
    }
    lemma_models = {
        4: build_ngram_model(lemma_tokens, n=4),
        3: build_ngram_model(lemma_tokens, n=3),
        2: build_ngram_model(lemma_tokens, n=2),
    }

    print(f"raw tokens: {len(raw_tokens)}, lemmatized tokens: {len(lemma_tokens)}")
    return {"raw": raw_models, "lemmatized": lemma_models}



def backoff_predictor(history, models):

    h4 = tuple(history[-3:])
    if len(h4) == 3 and h4 in models[4]:
        return sorted(models[4][h4].items(), key=lambda x: x[1], reverse=True)

    h3 = tuple(history[-2:])
    if len(h3) == 2 and h3 in models[3]:
        return sorted(models[3][h3].items(), key=lambda x: x[1], reverse=True)

    h2 = tuple(history[-1:])
    if len(h2) == 1 and h2 in models[2]:
        return sorted(models[2][h2].items(), key=lambda x: x[1], reverse=True)

    return []



def generate_text(seed_phrase, models, use_lemmatization=True, max_words=30):
    seed_clean = re.sub(r'[^a-z\s]', ' ', seed_phrase.lower())
    seed_words = word_tokenize(seed_clean)
    if use_lemmatization:
        seed_words = lemmatize_words(seed_words)

    output_tokens = ['<s>'] + seed_words   # <s> gives generation a starting context

    for _ in range(max_words):
        candidates = backoff_predictor(tuple(output_tokens), models)
        if not candidates:
            break

        candidates =  candidates[:5] if len(candidates) > 5  else candidates
        words, probs = zip(*candidates)
        probs = np.array(probs) / sum(probs) #normalize the probabilities
        next_word = np.random.choice(words, p=probs)

        if next_word == '</s>':
            break   # natural stop, one sentence generated

        output_tokens.append(next_word)

    generated = output_tokens[1:]
    return ' '.join(generated)

In [20]:
def run_generator():
    models = train_models()

    seed_phrase = input("Enter seed phrase (or press Enter for empty): ")

    raw_output = generate_text(seed_phrase, models["raw"], use_lemmatization=False)
    lemma_output = generate_text(seed_phrase, models["lemmatized"], use_lemmatization=True)

    print(f"\n[raw]        {raw_output}")
    print(f"[lemmatized] {lemma_output}")


In [21]:
if __name__ == '__main__':
    run_generator()

raw tokens: 0, lemmatized tokens: 0

[raw]        
[lemmatized] 
